# Phase 11 — Final Gate Review

**Status:** Complete  
**Workflow:** Notebook-first training documentation  
**Purpose:** Run a go/no-go review for model readiness using documented metrics, artifacts, risks, and product boundaries.

This notebook writes `reports/phase_11_final_gate_review.json` as the machine-readable final gate artifact. The current decision is intentionally conservative: the design documentation is complete, but release readiness remains blocked until trusted labels, full score coverage, calibration evidence, ATS benchmark results, candidate-set ranking labels, and export manifests exist.

## Review boundary

The final gate checks readiness evidence; it does not promote the model by default. A phase can be documentation-complete while still failing production readiness. Public or production use requires metrics from verified runs, not only templates or weak-label prototype reports.

Model/core may own calibrated scores and grounded signals. Backend/API wrapper remains owner of authorization, persistence, hydrated job details, product copy, and OpenAI wrapper orchestration.

## Shared setup

### Purpose
Load prior phase reports and API contract evidence needed for the final gate review.

### Required input
Repository root with `TODOS.md`, `GAP_MODEL_TRAINING.md`, `references/docs/generated/openapi.json`, and `reports/phase_00_reproducibility_snapshot.json` through `reports/phase_10_calibration_model_card.json`.

### Action
Read all prior report artifacts, collect acceptance flags, blockers, model/API boundaries, readiness decisions, and metric gate requirements.

### Expected output
Reusable dictionaries for prior phase reports, phase acceptance status, known blockers, output boundaries, and the Phase 11 report path.

### Verification
Fail fast if any required prior phase report is missing. Confirm every phase from 0 through 10 has acceptance evidence before the final decision is built.

In [4]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "TODOS.md").exists() and (candidate / "training").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook runtime.")


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / "reports"
OPENAPI_PATH = ROOT / "references" / "docs" / "generated" / "openapi.json"
REPORT_PATH = REPORTS / "phase_11_final_gate_review.json"


def read_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Required Phase 11 input is missing: {path.relative_to(ROOT)}")
    return json.loads(path.read_text())


phase_paths = {
    phase: REPORTS / f"phase_{phase:02d}_{name}.json"
    for phase, name in {
        0: "reproducibility_snapshot",
        1: "data_audit_contracts",
        2: "label_schema_baselines",
        3: "normalization_feature_design",
        4: "pair_generation_splits",
        5: "baseline_evaluation",
        6: "jobfit_training_experiments",
        7: "ats_friendliness_scoring",
        8: "overall_impression_signals",
        9: "candidate_reranking",
        10: "calibration_model_card",
    }.items()
}
phase_reports = {phase: read_json(path) for phase, path in phase_paths.items()}
openapi = read_json(OPENAPI_PATH)

acceptance_by_phase = {
    phase: report.get("acceptance", {})
    for phase, report in phase_reports.items()
}
phase_acceptance_complete = {
    phase: bool(flags) and all(flags.values())
    for phase, flags in acceptance_by_phase.items()
}

known_blockers = sorted(
    {
        blocker
        for report in phase_reports.values()
        for blocker in report.get("blocked_until_later_phases", [])
    }
    | set(phase_reports[5]["training_readiness_gate"]["blockers"])
    | set(phase_reports[10]["deployment_readiness"]["blockers_before_inference_release"])
)

setup_summary = {
    "phase_reports_loaded": len(phase_reports),
    "all_acceptance_complete": all(phase_acceptance_complete.values()),
    "known_blocker_count": len(known_blockers),
    "model_owned_field_count": len(phase_reports[1]["model_owned_fields"]),
    "wrapper_or_backend_field_count": len(phase_reports[1]["wrapper_or_backend_fields"]),
    "phase5_decision": phase_reports[5]["training_readiness_gate"]["decision"],
    "phase10_decision": phase_reports[10]["deployment_readiness"]["current_decision"],
}
setup_summary

{'phase_reports_loaded': 11,
 'all_acceptance_complete': True,
 'known_blocker_count': 40,
 'model_owned_field_count': 15,
 'wrapper_or_backend_field_count': 12,
 'phase5_decision': 'NO_GO_FIX_PAIR_GENERATION_AND_LABELS_FIRST',
 'phase10_decision': 'design_complete_deployment_not_ready'}

## Step 11.1 — Full pipeline review

### Purpose
Verify that data audit, labels, pairs, baselines, training, ATS scoring, recommendation reranking, calibration, and artifacts are complete.

### Required input
All prior phase reports and their acceptance flags.

### Action
Build a phase-by-phase review table that separates documentation completion from release readiness. Mark design-only phases as complete for planning but blocked for production when required evidence is missing.

### Expected output
A full pipeline review with component status, evidence sources, gate result, and blocking reason.

### Verification
Confirm every prior phase from 0 through 10 appears exactly once and each row has an evidence source.

## Step 11.2 — Gate metric review

### Purpose
Compare final metrics against required thresholds for job fit, ATS friendliness, recommendations, robustness, and language slices.

### Required input
Phase 5 baseline metrics, Phase 6 selection gates, Phase 7 ATS evaluation policy, Phase 9 ranking evaluation policy, and Phase 10 calibration diagnostics.

### Action
Create metric-gate rows for each output family. Use available metric evidence where it exists, and explicitly fail gates that only have design policy but no trusted evaluation data.

### Expected output
A gate metric review that identifies pass/fail status, current evidence, required threshold, and next action for each gate.

### Verification
Confirm job fit, ATS friendliness, recommendations, robustness/language slices, and calibration are all represented.

## Step 11.3 — Contract review

### Purpose
Confirm model outputs remain within product/API boundaries and do not own wrapper-only or backend-owned fields.

### Required input
Phase 1 boundary definition, model-owned fields, wrapper/backend fields, Phase 6 output signal contract, Phase 7 ATS output contract, Phase 8 grounded impression policy, Phase 9 candidate reranking contract, and OpenAPI schema evidence.

### Action
Record allowed model/core responsibilities, wrapper/backend responsibilities, contract checks, and release blockers. Treat design boundary as passing only when ownership is clear; implementation remains blocked until validators and wrapper checks exist.

### Expected output
A contract review with boundary status, allowed outputs, forbidden outputs, and required enforcement controls.

### Verification
Confirm wrapper-owned fields are not promoted into model/core ownership and backend-owned job hydration remains out of the model artifact.

## Step 11.4 — Risk review

### Purpose
Document remaining data quality, label quality, calibration, localization, and deployment risks.

### Required input
Phase 0 risk register, Phase 5 readiness blockers, Phase 7 ATS blockers, Phase 9 reranking blockers, Phase 10 deployment blockers, and language/slice gate policies from Phase 6.

### Action
Create a consolidated risk register with category, severity, evidence, impact, owner, and next control.

### Expected output
A final risk review that makes blocking risks visible before any staging or production claim.

### Verification
Confirm risk categories include data quality, label quality, calibration, localization, deployment, and contract boundary.

## Step 11.5 — Readiness decision

### Purpose
Choose one status: prototype-only, staging-ready, or production-ready, with evidence and required next actions.

### Required input
Pipeline review, metric gate review, contract review, risk review, Phase 5 training readiness decision, and Phase 10 deployment readiness decision.

### Action
Choose the safest readiness status supported by evidence. List allowed uses, blocked uses, evidence, and ordered next actions.

### Expected output
An evidence-based readiness decision with clear follow-up work.

### Verification
Confirm status is one of `prototype-only`, `staging-ready`, or `production-ready`; production/staging cannot pass while blocking gates remain.

## Report generation

### Purpose
Write the complete final gate review artifact.

### Required input
Shared setup constants and prior phase reports.

### Action
Build `reports/phase_11_final_gate_review.json` with pipeline review, metric gate review, contract review, risk review, readiness decision, blockers, next actions, and acceptance status.

### Expected output
`reports/phase_11_final_gate_review.json`.

### Verification
The report must satisfy all Phase 11 acceptance criteria and preserve conservative release gating.

In [5]:
phase_names = {
    0: "Reproducibility snapshot",
    1: "Data audit and output contracts",
    2: "Label schema and baseline definitions",
    3: "Normalization and feature design",
    4: "Pair generation and split strategy",
    5: "Baseline evaluation",
    6: "JobFitAlignment training experiments",
    7: "ATS friendliness scoring",
    8: "Overall impression signals",
    9: "Candidate job reranking",
    10: "Calibration, model card, and artifact manifest",
}

release_ready_by_phase = {
    0: False,
    1: False,
    2: False,
    3: False,
    4: False,
    5: False,
    6: False,
    7: False,
    8: False,
    9: False,
    10: False,
}

phase_gate_notes = {
    0: "Snapshot complete, but risks require later controls before release.",
    1: phase_reports[1]["data_readiness_decision"]["decision"],
    2: "Label and baseline schema complete; manual validation labels still required before production claims.",
    3: "Normalization and feature design complete; must be materialized into versioned configs before serving.",
    4: "Balanced pair design complete; materialized full-range pair artifact still required.",
    5: phase_reports[5]["training_readiness_gate"]["decision"],
    6: phase_reports[6]["selection_criteria"]["current_gate_result"]["decision"],
    7: phase_reports[7]["evaluation_policy"]["promotion_policy"]["current_decision"],
    8: phase_reports[8]["promotion_gate"]["current_decision"],
    9: phase_reports[9]["ranking_evaluation"]["current_readiness_decision"]["status"],
    10: phase_reports[10]["deployment_readiness"]["current_decision"],
}

pipeline_review = [
    {
        "phase": phase,
        "component": phase_names[phase],
        "report_path": str(phase_paths[phase].relative_to(ROOT)),
        "documentation_acceptance_complete": phase_acceptance_complete[phase],
        "release_ready": release_ready_by_phase[phase],
        "gate_note": phase_gate_notes[phase],
    }
    for phase in range(0, 11)
]

best_metrics = phase_reports[5]["training_readiness_gate"]["best_validation_metrics"]
selection = phase_reports[6]["selection_criteria"]

metric_gate_review = [
    {
        "gate": "jobFitAlignment regression metrics",
        "required_threshold": selection["minimum_global_gates"],
        "current_evidence": {
            "best_legacy_weak_label_baseline": best_metrics,
            "relative_mae_improvement_vs_constant_mean": phase_reports[5]["training_readiness_gate"]["relative_mae_improvement_vs_constant_mean"],
            "human_labeled_validation_available": False,
            "high_fit_validation_count": phase_reports[5]["training_readiness_gate"]["high_fit_validation_count"],
            "high_fit_test_count": phase_reports[5]["training_readiness_gate"]["high_fit_test_count"],
        },
        "gate_result": "fail_release_gate",
        "reason": "Metrics are available only for legacy weak labels and validation/test splits have no high-fit examples.",
        "next_action": "Materialize balanced pairs, add trusted validation labels, then rerun baseline and model comparison.",
    },
    {
        "gate": "ATS friendliness benchmark metrics",
        "required_threshold": phase_reports[7]["evaluation_policy"]["metrics"],
        "current_evidence": {
            "benchmark_labels_available": False,
            "required_case_families_defined": len(phase_reports[7]["benchmark_cases"]),
            "minimum_coverage_policy": phase_reports[7]["evaluation_policy"]["minimum_coverage"],
        },
        "gate_result": "fail_release_gate",
        "reason": "ATS issue labels and controlled benchmark results do not exist yet.",
        "next_action": "Create locked CV benchmark, label issue taxonomy, report precision/recall, bucket agreement, empty-text rate, and failure review.",
    },
    {
        "gate": "Recommendation reranking metrics",
        "required_threshold": phase_reports[9]["ranking_evaluation"]["required_metrics"],
        "current_evidence": {
            "backend_candidate_sets_available": False,
            "relevance_labels_available": False,
            "current_status": phase_reports[9]["ranking_evaluation"]["current_readiness_decision"],
        },
        "gate_result": "fail_release_gate",
        "reason": "NDCG/MAP cannot be production evidence without backend-like candidate sets and relevance labels.",
        "next_action": "Collect candidate-set ranking labels and enforce membership, uniqueness, score range, and max-item constraints.",
    },
    {
        "gate": "Robustness and language slices",
        "required_threshold": selection["slice_stability_gates"],
        "current_evidence": {
            "validation_score_bands": phase_reports[5]["training_readiness_gate"]["validation_score_bands"],
            "test_score_bands": phase_reports[5]["training_readiness_gate"]["test_score_bands"],
            "required_slices": selection["slice_stability_gates"]["required_slices"],
        },
        "gate_result": "fail_release_gate",
        "reason": "Current validation/test evidence is missing high-fit band coverage and audited slice metadata.",
        "next_action": "Report MAE, band agreement, and regression checks by role, language, experience band, pair type, and score band.",
    },
    {
        "gate": "Calibration and score semantics",
        "required_threshold": phase_reports[10]["calibration_diagnostics"]["metric_definitions"],
        "current_evidence": {
            "outputs_to_calibrate": phase_reports[10]["calibration_diagnostics"]["outputs_to_calibrate"],
            "current_blockers": phase_reports[10]["calibration_diagnostics"]["current_blockers"],
            "deployment_decision": phase_reports[10]["deployment_readiness"]["current_decision"],
        },
        "gate_result": "fail_release_gate",
        "reason": "Calibration tables, plots, bucket errors, and trusted labels are not available for promoted scores.",
        "next_action": "Run calibration diagnostics after verified model outputs, ATS labels, and candidate-set labels exist.",
    },
]

contract_review = {
    "boundary_status": "design_boundary_clear_release_enforcement_blocked",
    "model_owned_fields": phase_reports[1]["model_owned_fields"],
    "wrapper_or_backend_fields": phase_reports[1]["wrapper_or_backend_fields"],
    "allowed_model_core_responsibilities": [
        "Calibrated jobFitAlignment score and grounded matched/missing alignment signals.",
        "ATS friendliness score and detected issue keys after benchmark validation.",
        "Grounded overallImpression signal or template slots based only on observed evidence.",
        "Recommendation jobId, matchScore, matchLevel, matchedSkills, missingSkills, and rankingSignals for backend-provided candidates only.",
    ],
    "forbidden_model_core_responsibilities": [
        "topActionables product copy ownership",
        "sectionReviews product copy ownership",
        "job detail hydration or stale static job details",
        "auth, ownership, persistence, request validation, or OpenAI wrapper orchestration",
        "unsupported skills, seniority, hiring outcome, or ATS outcome claims",
    ],
    "required_enforcement_controls": [
        "Backend response validator for score range, recommendation membership, duplicate job IDs, and max item count.",
        "Wrapper evidence ledger checks before rendering public copy.",
        "Artifact manifest and model card hash validation before serving.",
        "Fallback policy for low confidence, missing target job, empty CV parse, unknown language, and sparse calibration buckets.",
    ],
    "review_result": "pass_design_contract_no_go_release_without_enforcement",
}

risk_review = [
    {
        "category": "data_quality",
        "severity": "blocking",
        "evidence": "Balanced Phase 4 pair metadata columns are missing and validation/test splits have no high-fit examples.",
        "impact": "Full 0-100 score range and high-fit behavior cannot be evaluated.",
        "owner": "model_training",
        "next_control": "Materialize balanced pair dataset with required metadata and high-fit validation/test coverage.",
    },
    {
        "category": "label_quality",
        "severity": "blocking",
        "evidence": "Legacy fit_score remains weak-label prototype evidence and human-labeled validation data is unavailable.",
        "impact": "Model can learn heuristics instead of trusted job-fit outcomes.",
        "owner": "model_training + reviewers",
        "next_control": "Create manual validation sample and label guideline before promotion metrics are trusted.",
    },
    {
        "category": "calibration",
        "severity": "blocking",
        "evidence": "Calibration bucket tables, plots, ECE, MCE, and score bucket errors are only defined, not populated from verified runs.",
        "impact": "Public score semantics could overstate reliability.",
        "owner": "model_training",
        "next_control": "Run calibration diagnostics for jobFitAlignment, atsFriendliness, and matchScore after trusted labels exist.",
    },
    {
        "category": "localization",
        "severity": "blocking",
        "evidence": "Language slice gates are defined, but release metrics by ID, EN, MIXED, and UNKNOWN slices are not available.",
        "impact": "Language-specific regressions or fallback failures can be hidden.",
        "owner": "model_training + product_review",
        "next_control": "Report slice metrics and manual review failures by language and text coverage.",
    },
    {
        "category": "deployment",
        "severity": "blocking",
        "evidence": "Artifact manifest hashes, serving bundle validation, model card fields, and runtime packaging are not populated from a verified export.",
        "impact": "Inference can fail or load unverifiable artifacts.",
        "owner": "model_training + platform",
        "next_control": "Export serving artifacts with immutable hashes, schema versions, model card, and runtime compatibility check.",
    },
    {
        "category": "contract_boundary",
        "severity": "blocking",
        "evidence": "Backend validators and wrapper grounding checks are required before hydrated jobs, recommendations, and product copy can be trusted.",
        "impact": "Model output could invent jobs, unsupported claims, or wrapper-owned fields if not enforced.",
        "owner": "backend_api + wrapper",
        "next_control": "Implement response validation, candidate membership checks, duplicate rejection, and evidence-backed copy rendering.",
    },
]

blocking_gate_count = sum(row["gate_result"] != "pass" for row in metric_gate_review)
readiness_status = "prototype-only"

readiness_decision = {
    "status": readiness_status,
    "status_options": ["prototype-only", "staging-ready", "production-ready"],
    "decision_summary": "Documentation and review artifacts are complete, but model readiness is prototype-only because all release metric gates remain blocked by missing trusted labels, missing high-fit coverage, missing ATS benchmark results, missing candidate-set ranking labels, and missing calibration/export evidence.",
    "allowed_uses": [
        "Planning next training work.",
        "Offline prototype analysis with explicit weak-label disclaimer.",
        "Generating review checklists, label guidelines, benchmark manifests, and export templates.",
    ],
    "blocked_uses": [
        "Production or staging scoring claims.",
        "Public score-band wording as calibrated quality evidence.",
        "ATS friendliness production claims.",
        "Recommendation reranking claims using static job artifacts.",
        "Automated deployment without manifest hashes and model card evidence.",
    ],
    "evidence": [
        f"Phase 5 training readiness decision: {phase_reports[5]['training_readiness_gate']['decision']}.",
        f"Phase 10 deployment decision: {phase_reports[10]['deployment_readiness']['current_decision']}.",
        "Validation/test splits have zero high-fit examples under the current high-band threshold.",
        "Human-labeled validation data is not available.",
        "ATS benchmark labels and recommendation relevance labels are not available.",
        "Calibration diagnostics and artifact manifest hashes are not populated from a verified model export.",
    ],
    "required_next_actions": [
        "Materialize Phase 4 balanced pair dataset with required metadata, leakage-safe splits, and high-fit validation/test coverage.",
        "Create human/manual validation labels and reviewer guidelines for job-fit score bands.",
        "Rerun Phase 5 baselines and Phase 6 job-fit experiments against trusted full-range labels and slices.",
        "Create ATS benchmark manifest with required file families, issue labels, precision/recall, empty-text review, and calibration evidence.",
        "Collect backend-like candidate sets with relevance labels and report NDCG@5, NDCG@10, MAP@10, and constraint violation rates.",
        "Run calibration diagnostics for all public score outputs with bucket counts, bucket errors, ECE, MCE, and confidence notes.",
        "Export serving artifacts with immutable sha256 hashes, schema versions, model card, artifact manifest, and runtime compatibility check.",
        "Implement backend/wrapper enforcement for candidate membership, duplicate rejection, score bounds, fallback policy, and grounded product copy.",
    ],
    "blocking_gate_count": blocking_gate_count,
}

acceptance = {
    "final_readiness_status_is_evidence_based": readiness_decision["status"] in readiness_decision["status_options"] and len(readiness_decision["evidence"]) >= 3,
    "remaining_risks_are_documented": {"data_quality", "label_quality", "calibration", "localization", "deployment", "contract_boundary"}.issubset({risk["category"] for risk in risk_review}),
    "next_actions_are_clear": len(readiness_decision["required_next_actions"]) >= 5 and all(action.endswith(".") for action in readiness_decision["required_next_actions"]),
}

report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_references": [
        "TODOS.md",
        "GAP_MODEL_TRAINING.md",
        "references/docs/generated/openapi.json",
        *[str(path.relative_to(ROOT)) for path in phase_paths.values()],
    ],
    "setup_summary": setup_summary,
    "pipeline_review": pipeline_review,
    "gate_metric_review": metric_gate_review,
    "contract_review": contract_review,
    "risk_review": risk_review,
    "readiness_decision": readiness_decision,
    "known_blockers": known_blockers,
    "acceptance": acceptance,
}

REPORTS.mkdir(exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
report["readiness_decision"]["status"], report["acceptance"]

('prototype-only',
 {'final_readiness_status_is_evidence_based': True,
  'remaining_risks_are_documented': True,
  'next_actions_are_clear': True})

## Acceptance criteria

- [x] Final readiness status is evidence-based.
- [x] Remaining risks are documented.
- [x] Next actions are clear.

## Verification

### Purpose
Confirm the saved report satisfies Phase 11 requirements.

### Required input
`reports/phase_11_final_gate_review.json`.

### Action
Read the report and assert phase coverage, metric-gate coverage, contract boundary status, risk categories, readiness status, and acceptance criteria.

### Expected output
A compact verification summary with report path, readiness status, gate count, risk categories, and acceptance flags.

### Verification
All assertions pass.

In [6]:
saved_report = read_json(REPORT_PATH)
phase_numbers = {row["phase"] for row in saved_report["pipeline_review"]}
gate_names = {row["gate"] for row in saved_report["gate_metric_review"]}
risk_categories = {risk["category"] for risk in saved_report["risk_review"]}

assert phase_numbers == set(range(0, 11))
assert len(saved_report["pipeline_review"]) == 11
assert {
    "jobFitAlignment regression metrics",
    "ATS friendliness benchmark metrics",
    "Recommendation reranking metrics",
    "Robustness and language slices",
    "Calibration and score semantics",
}.issubset(gate_names)
assert saved_report["contract_review"]["review_result"] == "pass_design_contract_no_go_release_without_enforcement"
assert {"data_quality", "label_quality", "calibration", "localization", "deployment", "contract_boundary"}.issubset(risk_categories)
assert saved_report["readiness_decision"]["status"] == "prototype-only"
assert saved_report["readiness_decision"]["blocking_gate_count"] >= 1
assert all(saved_report["acceptance"].values())

verification_summary = {
    "report_path": str(REPORT_PATH.relative_to(ROOT)),
    "readiness_status": saved_report["readiness_decision"]["status"],
    "pipeline_phase_count": len(saved_report["pipeline_review"]),
    "metric_gate_count": len(saved_report["gate_metric_review"]),
    "risk_categories": sorted(risk_categories),
    "acceptance": saved_report["acceptance"],
}
verification_summary

{'report_path': 'reports/phase_11_final_gate_review.json',
 'readiness_status': 'prototype-only',
 'pipeline_phase_count': 11,
 'metric_gate_count': 5,
 'risk_categories': ['calibration',
  'contract_boundary',
  'data_quality',
  'deployment',
  'label_quality',
  'localization'],
 'acceptance': {'final_readiness_status_is_evidence_based': True,
  'next_actions_are_clear': True,
  'remaining_risks_are_documented': True}}

## Phase notes

- Final gate status is `prototype-only`, not staging-ready or production-ready.
- Documentation artifacts for the notebook-first plan are complete through Phase 11.
- Release remains blocked by weak labels, no high-fit validation/test coverage, missing ATS benchmark labels, missing recommendation candidate-set labels, missing calibration diagnostics, and missing verified serving artifact manifest.
- Product/API boundary remains clear: backend/wrapper owns auth, persistence, hydrated job details, product copy, and OpenAI orchestration; model/core owns only calibrated scores and grounded signals after evidence exists.